# Course 5: Building a Small Bangla Language Model
## Session 10: Smaller and faster

**Goal:** measure size, CPU speed and quality at 32-bit, 16-bit and 8-bit, and set up distillation.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every code cell here was run once on a CPU (November 2026) using the small built-in sample. Cells marked TODO or NOT RUN need your data, a GPU or the network.

In [ ]:
SAMPLE_LINES = [
    "আমি বাংলায় গান গাই।",
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "বাংলা আমার মাতৃভাষা।",
    "আজ আকাশ খুব পরিষ্কার।",
    "ছোট মেয়েটি নদীর ধারে খেলছিল।",
    "বাবা বাজার থেকে মাছ কিনে আনলেন।",
    "মা রান্নাঘরে ভাত রান্না করছেন।",
    "আমরা সবাই মিলে স্কুলে যাই।",
    "বৃষ্টি হলে মাঠে পানি জমে।",
    "পাখিরা সকালে গান গায়।",
    "দাদু প্রতিদিন সন্ধ্যায় গল্প বলেন।",
    "ছেলেটি বই পড়তে খুব ভালোবাসে।",
    "নদীতে অনেক নৌকা চলছে।",
    "শীতকালে আমরা পিঠা খাই।",
    "ঢাকা বাংলাদেশের রাজধানী।",
    "আমার বন্ধু ভালো ফুটবল খেলে।",
]

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

def train_bpe(lines, vocab_size, specials=("[UNK]", "<|endoftext|>")):
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.NFC()
    tok.pre_tokenizer = pre_tokenizers.Metaspace()
    tok.decoder = decoders.Metaspace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=list(specials))
    tok.train_from_iterator(lines, trainer)
    return tok

In [ ]:
# A tiny model and tokenizer trained on the built-in sample, so this notebook runs anywhere.
# Replace with your own tokenizer file and checkpoint from sessions 4 and 6.
torch.manual_seed(0)
tok = train_bpe(SAMPLE_LINES * 20, 300)
EOT = tok.token_to_id("<|endoftext|>")
ids = []
for line in SAMPLE_LINES * 20:
    ids += tok.encode(line).ids + [EOT]
data = torch.tensor(ids)
cfg = Config(vocab_size=tok.get_vocab_size(), block_size=32, n_layer=2, n_head=2, d_model=64)
model = GPT(cfg)

def get_batch(data, block_size, batch_size):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(300):
    x, y = get_batch(data, cfg.block_size, 32)
    _, loss = model(x, y)
    opt.zero_grad(); loss.backward(); opt.step()
print("tiny model trained, final loss", round(loss.item(), 3))

### 1. Size arithmetic

In [ ]:
n = 6_852_608
for bits, b in [(32, 4), (16, 2), (8, 1)]:
    print(f"{bits}-bit: {n * b / 1e6:.1f} MB")

### 2. Three versions of the model

In [ ]:
import io, copy, time, warnings
warnings.filterwarnings("ignore")

def size_mb(m):
    buf = io.BytesIO(); torch.save(m.state_dict(), buf); return buf.tell() / 1e6

model.eval()
m32 = model
m16 = copy.deepcopy(model).half()
m8 = torch.ao.quantization.quantize_dynamic(copy.deepcopy(model), {nn.Linear}, dtype=torch.qint8)
for name, m in [("32-bit", m32), ("16-bit", m16), ("8-bit linear layers", m8)]:
    print(f"{name}: {size_mb(m):.3f} MB")

PyTorch prints a notice that `quantize_dynamic` is moving to the separate `torchao` package. It works in PyTorch 2.15; check the current documentation. In this tiny model the embedding table dominates, so the 8-bit file shrinks less than a quarter.

### 3. Quality and CPU speed, before and after

In [ ]:
@torch.no_grad()
def val_loss(m, n_batches=20):
    torch.manual_seed(1)
    return sum(m(*get_batch(data, cfg.block_size, 32))[1].item() for _ in range(n_batches)) / n_batches

@torch.no_grad()
def tokens_per_second(m, n_tokens=200):
    x = data[:cfg.block_size].unsqueeze(0)
    t0 = time.time()
    for _ in range(n_tokens):
        m(x)
    return n_tokens / (time.time() - t0)

for name, m in [("32-bit", m32), ("8-bit", m8)]:
    loss = val_loss(m)
    print(f"{name}: loss {loss:.4f}, perplexity {math.exp(loss):.3f}, {tokens_per_second(m):,.0f} forward passes/s on CPU")

Half precision is mainly for GPUs; many CPU operations are slow or unsupported at 16-bit, so test `m16` on a GPU runtime.

### 4. Distillation loss

In [ ]:
def distill_loss(student_logits, teacher_logits, T=2.0):
    s = F.log_softmax(student_logits / T, dim=-1)
    t = F.softmax(teacher_logits / T, dim=-1)
    return F.kl_div(s, t, reduction="batchmean") * T * T

a = torch.randn(8, 100)
print("same logits:", round(distill_loss(a, a).item(), 6), "| different:", round(distill_loss(torch.randn(8, 100), a).item(), 3))

### 5. Distil a one-layer student from the two-layer teacher

In [ ]:
def train_student(use_teacher, steps=200):
    torch.manual_seed(0)
    student = GPT(Config(vocab_size=cfg.vocab_size, block_size=cfg.block_size, n_layer=1, n_head=2, d_model=64))
    opt = torch.optim.AdamW(student.parameters(), lr=3e-3)
    for _ in range(steps):
        x, y = get_batch(data, cfg.block_size, 32)
        logits, loss = student(x, y)
        if use_teacher:
            with torch.no_grad():
                t_logits, _ = model(x)
            loss = 0.5 * loss + 0.5 * distill_loss(logits.view(-1, logits.size(-1)), t_logits.view(-1, logits.size(-1)))
        opt.zero_grad(); loss.backward(); opt.step()
    student.eval()
    return val_loss(student)

print("student alone:       ", round(train_student(False), 4))
print("student with teacher:", round(train_student(True), 4))

### TODO
- Repeat with your real model on your test file, and report size, speed and perplexity in one table.
- Compare a distilled student with an undistilled one at equal steps over three seeds. On the tiny sample the two are close; the gap is yours to measure at real scale.